# 아침 모델 운영 (야간선물 → 예상 시가·노란띠·믿을 정도 → 채점)

betterkym이 고정한 모델 규칙(`workbench/morning_card.py`, `workbench/e62_band.py`)을 **고치지 않고** 그대로 매일 돌리는 노트북이다.
여기에 있는 것은 그 바깥 일뿐이다.

1. **처음 자료 받기**: 모델이 읽는 자료 폴더(`workbench/live/<이름>/data/*.csv`)를 API로 처음부터 만든다.
2. **매일 단계**: 06:20 예상 시가·노란띠(07:30 카드) → 08:11 NXT 장전 값으로 믿을 정도 등급(08:11 카드) → 09:01 실제 시가로 채점 → 20:30 장 마감 자료 이어 받기.
3. **DB 저장**: 받은 원자료와 카드·채점 결과를 서버 DB `model` 영역에 넣는다(화면은 DB만 읽는다).
4. **DB에서 자료 폴더 다시 만들기**: 서버 폴더가 지워져도 DB로 되살린다.

서버(오라클)에서 `run_model.py`가 이 노트북의 함수를 불러 일정대로 돌린다. 처리 코드는 모두 이 노트북에 있다.

**종목 3구분** (`data/service_universe.csv`)
- 1구분(tier 1): 모델 설계에 쓰인 200종목(코스피 150 + 코스닥 50). 별도 표시 없음.
- 2구분(tier 2): 서비스 대상(코스피 전체 + 코스닥150) 중 나머지 755종목. 같은 규칙으로 계산하고 "모델 설계에 쓰이지 않은 종목" 경고를 단다.
- 3구분: 그 밖의 종목. 계산하지 않고 화면에서 "서비스 대상이 아니에요"로 보인다.

**설계 종목을 200종목으로 고정하는 이유**: 노란띠 폭 규칙과 믿을 정도 계수를 학습하는 "설계 종목"은 `e62_band.split_stocks`가
종목 목록을 고정 씨앗으로 섞어 반으로 나눠 정한다. 목록에 955종목을 넣으면 나누는 결과가 달라져 1구분 값도 바뀐다.
그래서 나누기는 언제나 원래 200종목 목록으로 하고, 그렇게 학습한 규칙을 955종목 모두에 적용한다(1구분 값은 betterkym 결과와 같다).

## 1. 설정
- `MODEL_ENV_FILE`: 열쇠 파일. 서버는 `~/overnight/.env`(KIS_APP_KEY, KIS_APP_SECRET, DART_API_KEY, GNEWS_APP_DB_URL, ALERT_TELEGRAM_*). 값에 따옴표를 쓰지 않는다(모델 코드가 큰따옴표만 벗긴다).
- 한국투자증권(KIS) 조회는 초당 `KIS_RPS`번을 넘지 않게 여러 갈래로 나눠 보낸다(계좌 한도 초당 20번).

In [ ]:
import fcntl, hashlib, importlib.util, json, os, shutil, sys, threading, time, traceback, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

MODEL = Path(os.environ.get('MODEL_DIR', Path.cwd())).resolve()      # 이 노트북이 있는 폴더(overnight_briefing/model)
sys.path[:0] = [p for p in (str(MODEL), str(MODEL / 'src')) if p not in sys.path]
LIVE = MODEL / 'workbench/live'                                       # 날짜별 자료 폴더(live_refresh와 같은 자리)
CARDS = MODEL / 'workbench/runs/claude_live_r1'                       # 카드·채점 파일(morning_card 기본 자리)
ENV_FILE = Path(os.environ.get('MODEL_ENV_FILE', MODEL.parents[1] / '.env')).expanduser()
TIER1_FILE = MODEL / 'data/universe.csv'                              # 모델 설계 200종목(betterkym)
SERVICE_FILE = MODEL / 'data/service_universe.csv'                    # 1·2구분 955종목
DB_ENV = 'GNEWS_APP_DB_URL'                                           # 서버 DB(model 영역에 쓰기)
BOOT_NAME = '00000000boot'      # 처음 자료 폴더 이름. 날짜 폴더보다 이름이 앞이라 live_refresh가 이어받을 폴더로 고른다
HISTORY_START = '20241201'      # 일봉 시작(선물 근월물 표의 첫 계약 20241213 직전부터)
NXT_START = '20250304'          # NXT 개장일. 설계 종목(믿을 정도 학습에 쓰는 107종목)은 이날부터
NXT_RECENT_DAYS = 45            # 나머지 종목은 최근 45거래일(종목별 08:11 오차 크기 계산에 최소 20일 필요)
NIGHT_MINUTE_DAYS = 400         # 야간선물 1분봉: KIS가 약 1년 치만 준다. 그 안에서 받을 수 있는 만큼
DART_START = '20261001'         # 공시 목록 시작(실적 공시 다음 날 주의 문구에는 최근 것만 필요)
KIS_RPS = 15
KIS_WORKERS = 12              # 서버→KIS 응답이 느려(약 0.6초) 여러 갈래로 보낸다. 초당 한도는 KIS_RPS가 지킨다
KEEP_FOLDERS = 14               # 날짜 자료 폴더는 최근 14개만 남긴다(전부 DB에 있다)
KST = 'Asia/Seoul'
CALENDAR_FILE = MODEL / 'market_calendar/special_days.csv'           # 시장 특별한 날 달력(web/data/special_days.csv와 같은 파일, 휴장일 이름 칸 포함)
OPEN_WAIT_MIN = 60              # 채점: 개장 시각부터 시가가 들어올 때까지 기다리는 최대 시간(분)
OPEN_POLL_SEC = 30              # 채점: 시가가 들어왔는지 확인하는 간격(초)


def read_env(path=ENV_FILE):
    env = {}
    if Path(path).exists():
        for line in Path(path).read_text(encoding='utf-8').splitlines():
            if '=' in line and not line.lstrip().startswith('#'):
                k, v = line.split('=', 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env


ENV = read_env()
now_kst = lambda: pd.Timestamp.now(tz=KST)
today = lambda: now_kst().strftime('%Y%m%d')
print('모델 폴더', MODEL, '/ 열쇠 파일', ENV_FILE, '있음' if ENV_FILE.exists() else '없음')

## 2. 종목 구분과 KIS 조회 도구
- `model_universe()`: 모델 자료 폴더의 `universe.csv`(Code·Name·Marcap·Market·Industry·group·etf 칸)를 955종목으로 만든다. 1구분은 betterkym 표 그대로, 2구분은 업종(group)만 서비스 표에서 채운다(모델 계산에는 쓰이지 않는 칸).
- `kis_call()`: betterkym의 `live_refresh.kis_get`(실패 시 다시 시도, 한도 초과 시 1초 쉼)을 그대로 쓰고, 여러 갈래에서 불러도 초당 `KIS_RPS`번을 넘지 않게 줄 세운다.
- `fast_premarket()`, `fast_stocks()`, `fast_opens()`: betterkym의 `refresh_premarket`, `refresh_stocks`, `check_open.opens`와 **같은 조회·같은 결과 모양**이고, 여러 갈래로 보내 빠를 뿐이다. 955종목을 08:11 직후 1분 안에 받으려고 바꿔 끼운다.

In [ ]:
from collect import KIS
from workbench import e62_band as e62
from workbench import live_panel as lp
from workbench import live_refresh as lr

TIER1 = sorted(pd.read_csv(TIER1_FILE, dtype={'Code': str}).Code)
SERVICE = pd.read_csv(SERVICE_FILE, dtype={'code': str})


def model_universe():
    t1 = pd.read_csv(TIER1_FILE, dtype={'Code': str})
    t2 = SERVICE[SERVICE.tier == 2]
    add = pd.DataFrame(dict(Code=t2.code, Name=t2.name, Marcap=np.nan, Market=t2.market, Industry='', group=t2.sector, etf=''))
    return pd.concat([t1, add], ignore_index=True)


class Rate:
    """여러 갈래가 함께 써도 초당 rps번을 넘지 않게 차례를 나눠 준다."""
    def __init__(self, rps):
        self.gap, self.lock, self.next = 1 / rps, threading.Lock(), 0.0

    def wait(self):
        with self.lock:
            now = time.monotonic()
            t = max(now, self.next)
            self.next = t + self.gap
        time.sleep(max(0.0, t - now))


RATE = Rate(KIS_RPS)


def kis_call(kis, path, tr, params):
    RATE.wait()
    return lr.kis_get(kis, path, tr, params)


def pmap(fn, items, workers=KIS_WORKERS, label=None):
    out, n = [], len(items)
    with ThreadPoolExecutor(workers) as ex:
        for i, r in enumerate(ex.map(fn, items), 1):
            out.append(r)
            if label and (i % 2000 == 0 or i == n):
                print(f'  {label} {i}/{n}', flush=True)
    return out


def premarket_rows(kis, code, day):
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-time-dailychartprice', 'FHKST03010230',
                   {'FID_COND_MRKT_DIV_CODE': 'NX', 'FID_INPUT_ISCD': code, 'FID_INPUT_HOUR_1': '081000', 'FID_INPUT_DATE_1': day,
                    'FID_PW_DATA_INCU_YN': 'N', 'FID_FAKE_TICK_INCU_YN': ''})
    return [dict(day=day, code=code, hhmmss=r['stck_cntg_hour'], price=float(r['stck_prpr']), volume=float(r['cntg_vol']))
            for r in res.get('output2') or [] if r.get('stck_bsop_date') == day and '080000' <= r['stck_cntg_hour'] <= '081000']


def fast_premarket(kis, df, codes, days):
    """live_refresh.refresh_premarket과 같은 결과(df, 새 줄 수)."""
    rows = sum(pmap(lambda p: premarket_rows(kis, *p), [(c, d) for d in days for c in codes], label='NXT 장전'), [])
    new = pd.DataFrame(rows, columns=['day', 'code', 'hhmmss', 'price', 'volume'])
    keys = ['day', 'code', 'hhmmss']
    new = new.drop_duplicates(keys)
    if new.empty:
        return df, 0
    return pd.concat([df, new[~new.set_index(keys).index.isin(df.set_index(keys).index)]], ignore_index=True), len(new)


def stock_rows(kis, code, start, end):
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': start, 'FID_INPUT_DATE_2': end,
                    'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
    return [dict(date=r['stck_bsop_date'], Open=float(r['stck_oprc']), Close=float(r['stck_clpr']), code=code)
            for r in res.get('output2') or [] if r.get('stck_bsop_date') and start <= r['stck_bsop_date'] <= end and float(r['stck_oprc']) > 0]


def fast_stocks(kis, df, codes, start, last_close_day):
    """live_refresh.refresh_stocks와 같은 결과. 한 번에 100줄까지라 기간이 길면 130일씩 나눠 받는다."""
    spans, a = [], pd.Timestamp(start)
    while a <= pd.Timestamp(last_close_day):
        b = min(a + pd.Timedelta(days=130), pd.Timestamp(last_close_day))
        spans.append((a.strftime('%Y%m%d'), b.strftime('%Y%m%d')))
        a = b + pd.Timedelta(days=1)
    rows = sum(pmap(lambda p: stock_rows(kis, p[0], *p[1]), [(c, s) for s in spans for c in codes], label='종목 일봉'), [])
    new = pd.DataFrame(rows, columns=['date', 'Open', 'Close', 'code']).drop_duplicates(['date', 'code'])
    if new.empty:
        return df
    old = df.set_index(['date', 'code']).index
    return pd.concat([df, new[~new.set_index(['date', 'code']).index.isin(old)]], ignore_index=True).sort_values(['code', 'date'])


def fast_opens(kis, codes, day):
    """check_open.opens와 같은 결과({종목: 그날 시가})."""
    def one(code):
        res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                       {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': day, 'FID_INPUT_DATE_2': day,
                        'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
        for r in res.get('output2') or []:
            if r.get('stck_bsop_date') == day and float(r.get('stck_oprc') or 0) > 0:
                return code, float(r['stck_oprc'])
        return code, None
    return {c: o for c, o in pmap(one, list(codes)) if o is not None}


_split = e62.split_stocks


def patch_model():
    """빠른 조회로 바꿔 끼우고, 설계 종목 나누기를 200종목 목록으로 고정한다(계산 규칙은 그대로)."""
    lr.refresh_premarket = fast_premarket
    lr.refresh_stocks = lambda kis, df, codes, start, last: fast_stocks(kis, df, codes, start, last)
    e62.split_stocks = lambda codes, dev: _split(TIER1, dev)

## 3. 처음 자료 받기 (한 번)
`workbench/live/00000000boot/`에 모델이 읽는 파일을 모두 만든다. 다음 날부터 `live_refresh`가 이 폴더를 이어받는다.

| 파일 | 받는 곳 | 기간 |
|---|---|---|
| futures_daily.csv | KIS 선물 일봉(주간 F, 야간 CM), 근월물 이어 붙이기(`collect.collect_futures`) | 2024-12-13 ~ |
| night_minutes.csv | KIS 야간선물 1분봉(`live_refresh.night_minutes`) | KIS가 주는 최근 약 1년 |
| universe_daily.csv | KIS 종목 일봉 시가·종가, 955종목 | 2024-12-01 ~ |
| universe_premarket.csv | KIS NXT 08:00~08:10 1분봉 | 설계 종목 107개: 2025-03-04 ~ / 나머지: 최근 45거래일 |
| kr_daily.csv, ks200_long.csv | 코스피200 지수(FinanceDataReader) | 2024-11 ~ / 2013-06 ~ |
| us_hourly.csv, vix_daily.csv | yfinance 미국 선물·환율 시간봉, VIX | 최근 730일 |
| investor_daily.csv | (투자자별 매매. 지금 규칙(S1·D08)에 쓰이지 않아 값이 빈 자리 표시 한 줄) | - |
| nxt_premarket.csv | (betterkym 개발용 14종목 원본. 같은 자료를 universe_premarket에 넣어 빈 표) | - |
| fred_release_dates.csv, fomc_dates.csv | 저장소 `data/` 그대로 | - |
| ../dart_filings.csv | DART 공시 목록(`live_refresh.refresh_dart`) | 2026-10-01 ~ |

모델 규칙이 실제로 쓰는 입력: 노란띠(S1)는 야간선물 일봉으로 만든 |야간 수익률|과 종목 일봉, 믿을 정도(D08)는 NXT 08:10 값·예상 시가 크기·평소 오차·회귀 설명력. 1분봉·미국 시간봉·VIX는 패널의 다른 칸(지금 규칙에서 안 씀)과 화면의 밤사이 흐름에 쓰인다.

In [ ]:
def last_finished_day(now=None):
    """주간장이 끝난 마지막 날짜(16:00 전이면 어제까지)."""
    now = now or now_kst()
    d = now if now.hour >= 16 else now - pd.Timedelta(days=1)
    return d.strftime('%Y%m%d')


def save_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'  {path.name}: {len(df):,}줄', flush=True)


def investor_placeholder():
    """투자자별 매매(investor_daily.csv) 자리. 지금 규칙(S1·D08)에 쓰이지 않아 값은 비운다.
    줄이 하나도 없으면 숫자 칸이 글자 칸으로 읽혀 패널 계산이 멈추므로, 값이 빈 숫자 한 줄을 둔다(어느 종목·날짜와도 맞지 않음)."""
    return pd.DataFrame([dict(date='19000101', code='000000', value=np.nan, frgn=np.nan, orgn=np.nan)])


def boot_ks200(start='2013-06-01'):
    import FinanceDataReader as fdr
    k = fdr.DataReader('KS200', start).reset_index()
    k['date'] = pd.to_datetime(k.iloc[:, 0]).dt.strftime('%Y%m%d')
    kr = pd.DataFrame(dict(date=k.date, Open=k.Open, High=k.High, Low=k.Low, Close=k.Close, Volume=k.Volume, code='KS200', name='코스피200 지수'))
    long = k[['date', 'Open', 'Close']].rename(columns={'date': 'Date'})
    return kr[kr.date >= '20241101'].reset_index(drop=True), long


def boot_vix():
    import yfinance as yf
    d = yf.download('^VIX', period='730d', interval='1d', progress=False, auto_adjust=False)
    d.columns = [c[0] if isinstance(c, tuple) else c for c in d.columns]
    d.index = pd.to_datetime(d.index).strftime('%Y%m%d')
    return d[['Close']].rename_axis('us_date').reset_index()


def boot_night_minutes(kis, fut, until):
    nights = fut[(fut.session == 'night') & (fut.date <= until)].sort_values('date')
    nights = nights[nights.date >= (pd.Timestamp(until) - pd.Timedelta(days=NIGHT_MINUTE_DAYS)).strftime('%Y%m%d')]
    def one(r):
        RATE.wait()
        return lr.night_minutes(kis, r.date, r.code)
    frames = [f for f in pmap(one, list(nights.itertuples()), workers=3, label='야간 1분봉') if len(f)]
    nm = pd.concat(frames, ignore_index=True).sort_values(['date', 'hhmmss'], kind='stable') if frames else \
        pd.DataFrame(columns=['date', 'code', 'hhmmss', 'price', 'volume'])
    print(f'  야간 1분봉: {nm.date.nunique()}밤 받음({nm.date.min()} ~ {nm.date.max()}), 요청한 밤 {len(nights)}', flush=True)
    return nm


def bootstrap(until=None, name=BOOT_NAME, steps=None):
    """처음 자료 폴더 만들기. steps로 일부만 다시 받을 수 있다(예: steps={'premarket'})."""
    until = until or last_finished_day()
    out, data = LIVE / name, LIVE / name / 'data'
    data.mkdir(parents=True, exist_ok=True)
    want = lambda s: steps is None or s in steps
    kis = KIS(str(ENV_FILE))
    started = now_kst()
    uni = model_universe()
    codes = uni.Code.tolist()
    save_csv(uni, data / 'universe.csv')
    for n in ('fred_release_dates.csv', 'fomc_dates.csv'):
        shutil.copy2(MODEL / 'data' / n, data / n)
    if want('futures'):
        import collect
        fut = collect.collect_futures(str(ENV_FILE), until)
        fut = fut[[lr.finished(s, d, started) for s, d in zip(fut.session, fut.date)]]
        save_csv(fut, data / 'futures_daily.csv')
    fut = pd.read_csv(data / 'futures_daily.csv', dtype=lp.TYPES)
    if want('night'):
        save_csv(boot_night_minutes(kis, fut, until), data / 'night_minutes.csv')
    if want('stocks'):
        empty = pd.DataFrame(columns=['date', 'Open', 'Close', 'code'])
        save_csv(fast_stocks(kis, empty, codes, HISTORY_START, until), data / 'universe_daily.csv')
    if want('premarket'):
        days = sorted(fut[(fut.session == 'day') & (fut.date >= NXT_START) & (fut.date <= until)].date.unique())
        empty = pd.DataFrame(columns=['day', 'code', 'hhmmss', 'price', 'volume'])
        dev = json.loads((MODEL / 'workbench/protocol_i24_features.json').read_text())['development_codes']
        design, _ = _split(TIER1, dev)      # 믿을 정도(D08)는 설계 종목의 과거 행으로만 학습한다
        rest = [c for c in codes if c not in set(design)]
        pm, _ = fast_premarket(kis, empty, design, days)
        pm, _ = fast_premarket(kis, pm, rest, days[-NXT_RECENT_DAYS:])
        save_csv(pm.sort_values(['day', 'code', 'hhmmss']), data / 'universe_premarket.csv')
        save_csv(pd.DataFrame(columns=['date', 'day', 'code', 'hhmmss', 'price', 'volume']), data / 'nxt_premarket.csv')
    if want('market'):
        kr, long = boot_ks200()
        save_csv(kr, data / 'kr_daily.csv')
        save_csv(long, data / 'ks200_long.csv')
        import collect
        save_csv(collect.collect_us_hourly(), data / 'us_hourly.csv')
        save_csv(boot_vix(), data / 'vix_daily.csv')
        save_csv(investor_placeholder(), data / 'investor_daily.csv')
    if want('dart'):
        cols = ['code', 'corp_code', 'rcept_no', 'rcept_dt', 'report_nm', 'corp_cls', 'flr_nm', 'rm']
        empty = out / 'dart_empty.csv'
        pd.DataFrame(columns=cols).to_csv(empty, index=False)
        filings, n = lr.refresh_dart(str(ENV_FILE), set(codes), DART_START, until, old_path=empty)
        empty.unlink()
        save_csv(filings, out / 'dart_filings.csv')
    log = dict(started_at=started.isoformat(timespec='seconds'), kind='bootstrap', base='API에서 처음 받음',
               trading_day=dict(date=until, open=True, checked_by='bootstrap'),
               futures_last={s: fut[fut.session == s].date.max() for s in ('day', 'night')},
               inputs_sha256={n: lp.sha(data / n) for n in lp.INPUTS if (data / n).exists()},
               finished_at=now_kst().isoformat(timespec='seconds'))
    (out / 'refresh_manifest.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))
    missing = [n for n in lp.INPUTS if not (data / n).exists()]
    print('처음 자료 끝', until, '빠진 파일:', missing or '없음', flush=True)
    return out

## 4. DB 저장
서버 DB의 `model` 영역에 표를 만들고, 자료 폴더의 파일과 카드·채점 결과를 넣는다. 같은 열쇠(날짜·종목·시각)는 새 값으로 덮어써서 몇 번을 넣어도 같다.

| 표 | 내용 |
|---|---|
| model.stocks | 955종목과 구분(tier 1·2) |
| model.futures_daily, model.night_minutes | 야간선물 일봉·1분봉 |
| model.stock_daily, model.premarket, model.ks200 | 종목 일봉 시가·종가, NXT 장전 1분봉, 코스피200 |
| model.filings | DART 공시 목록 |
| model.cards, model.card_stocks | 아침 카드(07:30·08:11)와 종목별 예상 시가 H·노란띠 80%/90%·믿을 정도 |
| model.checks, model.check_stocks | 개장 뒤 채점(띠 안에 들었는지 등) |
| model.runs | 단계별 실행 기록(성공·실패) |
| model.intraday | 사용자 보유·관심 종목의 장중 1분봉(종목 상세 '1일' 그래프) |

In [ ]:
import psycopg

# model 영역은 서버 관리자 계정이 한 번 만들어 둔다(gnews_app 소유, gnews_read 읽기):
#   create schema model authorization gnews_app; grant usage on schema model to gnews_read;
#   alter default privileges for role gnews_app in schema model grant select on tables to gnews_read;
DDL = """
create table if not exists model.stocks (code text primary key, name text, market text, sector text, tier int);
create table if not exists model.futures_daily (date text, session text, code text, open float8, high float8, low float8, close float8, volume float8, primary key (date, session));
create table if not exists model.night_minutes (date text, code text, hhmmss text, price float8, volume float8, primary key (date, code, hhmmss));
create table if not exists model.stock_daily (code text, date text, open float8, close float8, primary key (code, date));
create table if not exists model.premarket (day text, code text, hhmmss text, price float8, volume float8, primary key (day, code, hhmmss));
create table if not exists model.ks200 (date text primary key, open float8, high float8, low float8, close float8, volume float8);
create table if not exists model.filings (rcept_no text primary key, code text, corp_code text, rcept_dt text, report_nm text, corp_cls text, flr_nm text, rm text);
create table if not exists model.cards (t text, slot text, status text, reason text, reason_text text, d text, x float8, issued_at timestamptz,
  timing_status text, readiness jsonb, grade_status jsonb, cautions jsonb, n_stocks int, n_graded int, file text, sha256 text, card jsonb, primary key (t, slot));
create table if not exists model.card_stocks (t text, slot text, code text, name text, pc float8, h float8, l80 float8, u80 float8, l90 float8, u90 float8,
  grade_level text, grade_p float8, nxt_g float8, cautions jsonb, primary key (t, slot, code));
create table if not exists model.checks (t text, slot text, scored int, in_band80 float8, in_band90 float8, mae_h float8, dir_nxt_all float8, summary jsonb, primary key (t, slot));
alter table model.checks add column if not exists checked_at timestamptz;
create table if not exists model.check_stocks (t text, slot text, code text, open float8, y float8, in80 bool, in90 bool, above bool, below bool, primary key (t, slot, code));
create table if not exists model.runs (id bigserial primary key, step text, t text, started_at timestamptz, finished_at timestamptz, ok bool, message text);
create index if not exists card_stocks_code on model.card_stocks (code, t);
create table if not exists model.us_daily (symbol text, us_date text, name text, open float8, high float8, low float8, close float8, prev_close float8,
  updated_at timestamptz default now(), primary key (symbol, us_date));
create table if not exists model.us_intraday (symbol text, us_date text, ts timestamptz, price float8, primary key (symbol, ts));
create table if not exists model.intraday (day text, code text, hhmm text, price float8, volume float8, primary key (day, code, hhmm));
"""


def db_url():
    url = os.environ.get(DB_ENV) or ENV.get(DB_ENV)
    if not url:
        raise SystemExit(f'{DB_ENV}가 열쇠 파일에 없다')
    return url.replace('postgresql+psycopg://', 'postgresql://')


def db():
    return psycopg.connect(db_url(), autocommit=False)


def ensure_tables():
    with db() as con:
        con.execute(DDL)


def upsert(table, df, keys, chunk=50000):
    """df를 임시 표로 복사한 뒤 table에 넣는다(같은 열쇠는 새 값으로)."""
    if df is None or df.empty:
        return 0
    cols = list(df.columns)
    rest = [c for c in cols if c not in keys]
    clean = df.astype(object).where(pd.notna(df), None)
    with db() as con, con.cursor() as cur:
        cur.execute(f'create temp table _up (like {table} including defaults) on commit drop')
        for i in range(0, len(clean), chunk):
            with cur.copy(f'copy _up ({", ".join(cols)}) from stdin') as cp:
                for row in clean.iloc[i:i + chunk].itertuples(index=False):
                    cp.write_row(row)
        upd = ', '.join(f'{c} = excluded.{c}' for c in rest) if rest else None
        cur.execute(f'insert into {table} ({", ".join(cols)}) select {", ".join(cols)} from _up '
                    f'on conflict ({", ".join(keys)}) do ' + (f'update set {upd}' if upd else 'nothing'))
        return len(clean)


def save_folder_db(folder, since=None):
    """자료 폴더의 원자료를 DB에 넣는다. since(YYYYMMDD)가 있으면 그날부터만."""
    ensure_tables()
    folder = Path(folder)
    data = folder / 'data'
    read = lambda n: pd.read_csv(data / n, dtype=lp.TYPES) if (data / n).exists() else None
    cut = lambda df, col: df if since is None or df is None else df[df[col] >= since]
    done = {}
    done['stocks'] = upsert('model.stocks', SERVICE[['code', 'name', 'market', 'sector', 'tier']], ['code'])
    f = read('futures_daily.csv')
    done['futures_daily'] = upsert('model.futures_daily', cut(f, 'date'), ['date', 'session'])
    nm = read('night_minutes.csv')
    done['night_minutes'] = upsert('model.night_minutes', cut(nm, 'date'), ['date', 'code', 'hhmmss'])
    ud = read('universe_daily.csv')
    if ud is not None:
        done['stock_daily'] = upsert('model.stock_daily', cut(ud, 'date').rename(columns={'Open': 'open', 'Close': 'close'})[['code', 'date', 'open', 'close']], ['code', 'date'])
    pm = read('universe_premarket.csv')
    done['premarket'] = upsert('model.premarket', cut(pm, 'day'), ['day', 'code', 'hhmmss'])
    kr = read('kr_daily.csv')
    if kr is not None:
        k = cut(kr[kr.code == 'KS200'], 'date')
        done['ks200'] = upsert('model.ks200', k[['date', 'Open', 'High', 'Low', 'Close', 'Volume']].set_axis(['date', 'open', 'high', 'low', 'close', 'volume'], axis=1), ['date'])
    if (folder / 'dart_filings.csv').exists():
        fl = pd.read_csv(folder / 'dart_filings.csv', dtype=str)
        done['filings'] = upsert('model.filings', cut(fl, 'rcept_dt'), ['rcept_no'])
    print('DB 저장', folder.name, {k: v for k, v in done.items() if v}, flush=True)
    return done


def save_card_db(path):
    path = Path(path)
    card = json.loads(path.read_text())
    sha = Path(str(path) + '.sha256').read_text().split()[0] if Path(str(path) + '.sha256').exists() else None
    t, slot = card['T'], card.get('slot')
    stocks = card.get('stocks', [])
    row = pd.DataFrame([dict(t=t, slot=slot, status=card.get('status'), reason=card.get('reason'), reason_text=card.get('reason_text'), d=card.get('D'),
                             x=(card.get('market') or {}).get('x'), issued_at=card.get('issued_at'), timing_status=card.get('timing_status'),
                             readiness=json.dumps(card.get('data_readiness'), ensure_ascii=False), grade_status=json.dumps(card.get('grade_status'), ensure_ascii=False),
                             cautions=json.dumps((card.get('market') or {}).get('cautions', []), ensure_ascii=False), n_stocks=len(stocks),
                             n_graded=sum(s.get('grade_0811') is not None for s in stocks), file=path.name, sha256=sha,
                             card=json.dumps({k: v for k, v in card.items() if k not in ('stocks', 'inputs')}, ensure_ascii=False))])
    ensure_tables()
    upsert('model.cards', row, ['t', 'slot'])
    rows = [dict(t=t, slot=slot, code=s['code'], name=s['name'], pc=s['base_close']['value'], h=s['H'], l80=s['band80'][0], u80=s['band80'][1],
                 l90=s['band90'][0], u90=s['band90'][1], grade_level=(s.get('grade_0811') or {}).get('level'), grade_p=(s.get('grade_0811') or {}).get('p'),
                 nxt_g=(s.get('grade_0811') or {}).get('nxt_g'), cautions=json.dumps(s.get('cautions', []), ensure_ascii=False)) for s in stocks]
    with db() as con:
        con.execute('delete from model.card_stocks where t = %s and slot = %s', (t, slot))
    upsert('model.card_stocks', pd.DataFrame(rows), ['t', 'slot', 'code'])
    print('카드 DB 저장', t, slot, card.get('status'), '종목', len(rows), flush=True)


def save_check_db(path, checked_at=None):
    """채점 파일을 DB에 넣는다. 종목 줄(check_stocks)을 먼저 넣고 표 줄(checks)을 나중에 넣는다(화면은 checks 줄이 생기면 채점을 띄운다).
    checked_at: 채점을 마친 시각(화면 공개 시각). 지난 날짜 채우기는 None."""
    c = json.loads(Path(path).read_text())
    slot = '0811' if '_0811' in c['card'] else '0730'
    ensure_tables()
    rows = pd.DataFrame(c['rows'])
    if len(rows):
        rows = rows.assign(t=c['date'], slot=slot)[['t', 'slot', 'code', 'open', 'y', 'in80', 'in90', 'above', 'below']]
        upsert('model.check_stocks', rows, ['t', 'slot', 'code'])
    upsert('model.checks', pd.DataFrame([dict(t=c['date'], slot=slot, scored=c['stocks_scored'], in_band80=c['in_band80'], in_band90=c['in_band90'],
                                              mae_h=c['mae_H'], dir_nxt_all=c['dir_nxt_all'],
                                              summary=json.dumps({k: v for k, v in c.items() if k != 'rows'}, ensure_ascii=False, default=float),
                                              checked_at=checked_at)]), ['t', 'slot'])
    print('채점 DB 저장', c['date'], slot, '띠 안', c['in_band80'], '%', flush=True)

## 5. 매일 단계
betterkym의 운영 순서(`seal_morning.sh`, `after_close.sh`)와 같다. 시각은 서버 cron이 맞춘다(한국 시각).

| 단계 | 시각 | 하는 일 |
|---|---|---|
| `morning0730` | 06:20 | 끝난 야간장·공시 이어 받기 → 07:30 카드(예상 시가 H, 노란띠) → 지난밤 1분봉 → DB |
| `morning0811` | 08:11:30 | NXT 08:00~08:10 955종목 받기 → 08:11 카드(믿을 정도 등급으로 보정) → DB |
| `open` | 09:00 | 달력(`market_calendar/special_days.csv`)의 그날 개장 시각(보통 09:00, 수능일·새해 첫날 10:00)부터 30초마다 시가가 들어왔는지 확인 → 들어오면 바로 두 카드 채점(카드 해시가 같은지 먼저 확인) → DB(`checked_at`에 채점 시각). 화면은 채점이 저장되면 바로 띄운다. 휴장일은 건너뛰고, 개장 뒤 60분까지 시가가 없으면 실패 알림 |
| `close` | 20:30 | 그날 주간장·종목 일봉·코스피200 이어 받기(다음 날 아침의 바탕) → DB → 오래된 폴더 정리. 16:00~20:00 애프터마켓 중에는 KIS 종가 칸이 시간외 가격이라 끝난 뒤에 받는다(betterkym 16:05 기록은 11-12 카드에서 168종목 전일 종가가 시간외 가격이었음) |

휴장일이면 `live_refresh`가 휴장으로 기록하고 카드는 "KRX 휴장일" 상태로 남는다.

`backfill`(손으로 한 번, 2026-11-20): 서비스 시작 전 지난 20거래일의 07:30·08:11 카드를 같은 규칙으로 다시 만들고, DB 일봉의 그날 시가로 채점해 넣는다(보관·기록·종목 상세의 '최근 20거래일'을 채우려고). 이미 있는 날짜는 건너뛴다. 카드에는 `mode: backfill`과 안내 문구가 붙는다.

In [ ]:
def run_main(fn, argv):
    old = sys.argv
    sys.argv = ['x'] + [str(a) for a in argv]
    try:
        fn()
    finally:
        sys.argv = old


def newest_card(T, slot):
    files = sorted(CARDS.glob(f'card_{T}_{slot}*.json'), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


def make_card(T, slot):
    from workbench import morning_card
    run_main(morning_card.main, ['--date', T, '--slot', slot, '--root', LIVE / T, '--events', LIVE / T / 'dart_filings.csv'])
    path = newest_card(T, slot)
    save_card_db(path)
    return path


def step_morning(T, slot):
    patch_model()
    argv = ['--name', T, '--env', ENV_FILE, '--until', T, '--no-night']
    if slot == '0811':
        argv += ['--premarket-days', T]
    run_main(lr.main, argv)
    manifest = json.loads((LIVE / T / 'refresh_manifest.json').read_text())
    checked = ''
    if slot == '0730' and not manifest.get('skipped'):   # 카드 전에 어제 종가 검사(5-4)
        try:
            checked = ' · ' + verify_prev_close(T)
        except Exception as err:
            checked = f' · 어제 종가 검사 실패 {type(err).__name__}: {err}'
            telegram(f'[아침 모델] {T} 어제 종가 검사 실패(카드는 그대로 만듦): {err}')
    path = make_card(T, slot)
    if manifest.get('skipped'):
        return f'휴장일: {manifest["skipped"]}'
    if slot == '0730':
        run_main(lr.main, ['--name', T, '--env', ENV_FILE, '--until', T, '--night-only'])
    save_folder_db(LIVE / T, since=(pd.Timestamp(T) - pd.Timedelta(days=10)).strftime('%Y%m%d'))
    card = json.loads(path.read_text())
    return f'{slot} 카드 {card["status"]} 종목 {len(card["stocks"])} 등급 {sum(s.get("grade_0811") is not None for s in card["stocks"])}' + checked


def load_check_open():
    spec = importlib.util.spec_from_file_location('check_open', CARDS / 'check_open.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    mod.opens = fast_opens
    return mod


def market_day(T):
    """달력(special_days.csv)의 그날 줄. 파일이 없거나 표 범위 밖이면 None(보통 날로 본다)"""
    try:
        cal = pd.read_csv(CALENDAR_FILE, dtype=str).fillna('')
    except FileNotFoundError:
        return None
    row = cal[cal.date == T]
    return None if row.empty else row.iloc[0]


def step_open(T):
    """그날 개장 시각(달력: 보통 09:00, 수능일·새해 첫날 10:00)부터 시가가 들어오는 즉시 두 카드를 채점한다.
    삼성전자 시가 한 건으로 들어왔는지 OPEN_POLL_SEC마다 확인하고, 들어오면 카드 종목 시가를 한 번 받아 두 카드에 같이 쓴다.
    개장 뒤 OPEN_WAIT_MIN분이 지나도 시가가 없으면 실패로 남긴다(텔레그램 알림)."""
    day = market_day(T)
    if day is not None and day.krx_open == '0':
        return '휴장일이라 채점 없음'
    def pending():
        todo = []
        for slot in ('0730', '0811'):
            card, out = newest_card(T, slot), CARDS / f'check_{T}_{slot}.json'
            if card is not None and not out.exists() and json.loads(card.read_text()).get('status') == 'ok':
                todo.append((slot, card, out))
        return todo
    if not pending():
        return '채점할 카드 없음'
    open_at = pd.Timestamp(f"{T[:4]}-{T[4:6]}-{T[6:]} {(day.open if day is not None and day.open else '09:00')}", tz=KST)
    deadline = open_at + pd.Timedelta(minutes=OPEN_WAIT_MIN)
    if now_kst() < open_at:
        print('개장', f'{open_at:%H:%M}', '까지 기다림', flush=True)
        time.sleep((open_at - now_kst()).total_seconds())
    kis = KIS(str(ENV_FILE))
    while not fast_opens(kis, ['005930'], T):
        if now_kst() > deadline:
            raise RuntimeError(f'{deadline:%H:%M}까지 그날 시가가 들어오지 않았습니다')
        time.sleep(OPEN_POLL_SEC)
    print('시가 들어옴', f'{now_kst():%H:%M:%S}', flush=True)
    need0811 = day is None or day.premarket == '1'       # 개장 전 거래가 있는 날은 08:11 카드도 채점한다
    for k in range(20):                                  # 08:11 카드가 아직 만들어지는 중이면 최대 10분 기다린다
        todo = pending()
        if not need0811 or newest_card(T, '0811') is not None:
            break
        print('08:11 카드를 기다림', flush=True)
        time.sleep(OPEN_POLL_SEC)
    codes = sorted({s['code'] for _, card, _ in todo for s in json.loads(card.read_text())['stocks']})
    for k in range(5):                                   # 거의 다 들어올 때까지(빠진 종목 1% 이하) 조금 더 기다린다
        op = fast_opens(kis, codes, T)
        if len(op) >= len(codes) * 0.99 or now_kst() > deadline:
            break
        print('시가', len(op), '/', len(codes), '· 조금 뒤 다시', flush=True)
        time.sleep(OPEN_POLL_SEC)
    mod = load_check_open()
    mod.opens = lambda kis, cs, d: {c: op[c] for c in cs if c in op}
    done = []
    for slot, card, out in todo:
        run_main(mod.main, ['--date', T, '--env', ENV_FILE, '--card', card, '--out', out])
        save_check_db(out, checked_at=now_kst().to_pydatetime())
        done.append(f'{slot} 채점')
    return f"{', '.join(done)} (시가 {len(op)}/{len(codes)}종목, {now_kst():%H:%M:%S})"


def cleanup_folders():
    names = sorted(p for p in LIVE.iterdir() if p.is_dir() and (p / 'refresh_manifest.json').exists())
    for p in names[:-KEEP_FOLDERS]:
        shutil.rmtree(p)
    return len(names[:-KEEP_FOLDERS])


BACKFILL_DAYS = 600     # 지난 거래일을 자료가 허락하는 만큼(일봉 2024-12~, 앞쪽 60거래일은 학습 자료가 모자라 건너뜀)


def backfill_panels(root):
    """지난 날짜용 자료 패널을 한 번만 만든다. 지난 날짜는 주간장 줄이 이미 있어 날짜를 넘겨도 패널이 같다(live_panel.add_today)."""
    p6, _ = lp.build(root, '06:00', None)
    p8, _ = lp.build(root, '08:11', None)
    dev = json.loads((lp.RESEARCH / 'workbench/protocol_i24_features.json').read_text())['development_codes']
    return p6, p8, pd.read_csv(root / 'dart_filings.csv', dtype=str), dev


def backfill_card(T, slot, panels):
    """지난 날짜 T의 카드를 같은 규칙으로 다시 만든다(2026-11-20, 서비스 시작 전 날짜 채우기).
    live 단계와 같은 자료 패널과 build_card를 쓰고, 계수는 그날 이전 행으로만 학습한다(morning_card 설명). 그날 시가는 입력에서 지운다."""
    from workbench import morning_card as mc
    from common import ACCOUNTS
    p6, p8, filings, dev = panels
    z, n, dev, codes = lp.frames_from_panels(p6, p8, filings, dev, keep_T=T)
    signal = lp.signal_of(p6, T)
    has_nxt = slot == '0811' and bool((n['T'] == T).any())
    card = mc.build_card(z, n, dev, codes, T, ACCOUNTS, False, signal=signal, slot=slot, allow_current_grade=has_nxt)
    card['slot'] = slot
    card['mode'] = 'backfill'
    card['issued_at'] = pd.Timestamp(f'{T} {slot[:2]}:{slot[2:]}', tz='Asia/Seoul').isoformat(timespec='seconds')   # 그날 목표 시각으로 적는다
    card.update(mc.issuance_timing(T, slot, card['issued_at'], historical_replay=True))
    card['notes'] = [*card.get('notes', []), '지난 날짜를 나중에 같은 규칙으로 다시 계산한 카드예요(2026-11-20 채움). 그날 이전 자료로만 계산했어요.']
    if slot == '0811' and not has_nxt:
        card['grade_status'] = dict(status='unavailable', reason='no_premarket', reason_text='그날 장전 거래 자료가 없어 믿을 정도를 내지 못했어요')
    out = CARDS / f'card_{T}_{slot}.json'
    text = json.dumps(card, ensure_ascii=False, indent=1, allow_nan=False)
    out.write_text(text)
    Path(str(out) + '.sha256').write_text(hashlib.sha256(text.encode()).hexdigest() + '  ' + out.name + '\n')
    save_card_db(out)
    return out, card


def db_opens(kis, codes, day):
    """check_open.opens와 같은 결과({종목: 그날 시가})를 KIS 대신 DB 일봉(model.stock_daily)에서 읽는다(지난 날짜 채점용)."""
    with db() as con:
        rows = con.execute('select code, open from model.stock_daily where date = %s and code = any(%s) and open > 0', (day, list(codes))).fetchall()
    return {c: float(o) for c, o in rows}


def fill_premarket_gaps(root, days):
    """최근 NXT_RECENT_DAYS 거래일 중 장전 거래(08:00~08:10)가 하나도 없는 날을 KIS에서 받아 자료 폴더와 DB에 채운다(예: 11-19). 채운 날짜 목록을 돌려준다."""
    path = root / 'data' / 'universe_premarket.csv'
    pm = pd.read_csv(path, dtype={'day': str, 'code': str, 'hhmmss': str})
    have = set(pm.day.unique())
    recent = sorted(days)[-NXT_RECENT_DAYS:]
    gaps = [d for d in recent if d not in have]
    if not gaps:
        return []
    kis = KIS(str(ENV_FILE))
    codes = model_universe().Code.tolist()
    pm2, n = fast_premarket(kis, pm, codes, gaps)
    if n:
        save_csv(pm2.sort_values(['day', 'code', 'hhmmss']), path)
        upsert('model.premarket', pm2[pm2.day.isin(gaps)], ['day', 'code', 'hhmmss'])
    print('장전 거래 채움', gaps, n, '줄', flush=True)
    return [d for d in gaps if (pm2.day == d).any()]


def us_daily_history(start='20241201'):
    """미국 지표 일봉을 start부터 채운다(지난 날짜 화면의 '미국 시장' 칸용). 이미 있는 날은 덮어써도 같은 값."""
    import yfinance as yf
    out = []
    for sym, name in US_SYMBOLS.items():
        d = yf.download(sym, start=pd.Timestamp(start).strftime('%Y-%m-%d'), interval='1d', progress=False, auto_adjust=False)
        d.columns = [c[0] if isinstance(c, tuple) else c for c in d.columns]
        if d.empty:
            continue
        d.index = pd.to_datetime(d.index).strftime('%Y%m%d')
        dd = d[['Open', 'High', 'Low', 'Close']].astype(float).assign(prev=lambda x: x.Close.shift(1)).dropna(subset=['prev'])
        upsert('model.us_daily', pd.DataFrame(dict(symbol=sym, us_date=dd.index, name=name, open=dd.Open, high=dd.High, low=dd.Low, close=dd.Close,
                                                  prev_close=dd.prev)).reset_index(drop=True), ['symbol', 'us_date'])
        out.append(f'{name} {len(dd)}일')
    return ', '.join(out)


def step_backfill(T=None):
    """T(기본: 오늘) 이전 BACKFILL_DAYS 거래일의 07:30·08:11 카드와 채점을 채운다(최근 날부터). 이미 DB에 있는 날짜·시각은 건너뛴다.
    장전 거래가 없는 날은 07:30 카드만 만든다. 구간 학습 자료가 60거래일 미만인 앞쪽 날짜는 건너뛴다. 미국 지표 일봉도 함께 채운다."""
    patch_model()
    T = T or today()
    with db() as con:
        days = [r[0] for r in con.execute('select distinct date from model.stock_daily where date < %s order by date desc limit %s', (T, BACKFILL_DAYS)).fetchall()]
        have = {(r[0], r[1]) for r in con.execute('select t, slot from model.cards').fetchall()}
        nograde = {r[0] for r in con.execute("select t from model.cards where slot = '0811' and n_graded = 0").fetchall()}
    root = sorted(p for p in LIVE.iterdir() if p.is_dir() and (p / 'refresh_manifest.json').exists())[-1]   # 가장 최근 자료 폴더
    print('미국 지표 일봉', us_daily_history(), flush=True)
    filled = set(fill_premarket_gaps(root, days))
    redo = {(d, '0811') for d in filled & nograde}                       # 장전 거래를 새로 채운 날은 08:11 카드를 다시
    todo = [(d, s) for d in sorted(days, reverse=True) for s in ('0730', '0811') if (d, s) not in have or (d, s) in redo]
    if not todo:
        return '채울 날짜 없음'
    print('자료 패널 만드는 중(한 번만)', root.name, '· 할 일', len(todo), flush=True)
    panels = backfill_panels(root)
    pm_days = set(panels[1].loc[panels[1].nxt.astype(bool), 'T'].unique())   # 장전 거래가 있는 날
    mod = load_check_open()
    mod.opens = db_opens
    done, skipped = [], []
    for day, slot in todo:
        if slot == '0811' and day not in pm_days:
            continue                                                           # 장전 거래가 없는 날은 07:30 카드만
        try:
            path, card = backfill_card(day, slot, panels)
        except SystemExit as err:                                              # 학습 자료 60거래일 미만 등
            skipped.append(day)
            print('건너뜀', day, slot, str(err)[:80], flush=True)
            continue
        if card.get('status') == 'ok':
            out = CARDS / f'check_{day}_{slot}.json'
            run_main(mod.main, ['--date', day, '--env', ENV_FILE, '--card', path, '--out', out])
            save_check_db(out)
        done.append(f"{day} {slot} {card.get('status')}")
        print('채움', done[-1], flush=True)
    return f'{len(done)}개 채움, 건너뜀 {len(set(skipped))}일({days[-1] if days else "-"} ~ {days[0] if days else "-"}, 자료 폴더 {root.name})'


def step_close(T):
    patch_model()
    run_main(lr.main, ['--name', T + 'c', '--env', ENV_FILE, '--until', T])
    folder = LIVE / (T + 'c')
    if json.loads((folder / 'refresh_manifest.json').read_text()).get('skipped'):
        shutil.rmtree(folder)
        return '휴장일'
    save_folder_db(folder, since=(pd.Timestamp(T) - pd.Timedelta(days=10)).strftime('%Y%m%d'))
    try:   # 사용자 종목 마감 봉('1일' 그래프 끝 = 공식 종가). 15:45엔 아직 없어서 여기서 다시(5-4)
        bars = refill_close_bars(T)
    except Exception as err:
        bars = f'마감 봉 다시 받기 실패 {type(err).__name__}: {err}'
    return f'장 마감 자료 저장, 지운 옛 폴더 {cleanup_folders()}개 · {bars}'   # 시세 요약은 15:45 close_price에서(5-3·5-4)

## 5-1. 밤사이 화면용 실시간 자료
- `night_live`(한국 18:00~06:10, 10분마다): 지금 진행 중인 야간선물 1분봉과 그날 주간 종가를 DB에 넣는다. 07:30 전·주말 화면의 "진행 중" 흐름용이다. 모델 계산은 06:20 단계가 자료 폴더로 따로 한다.
- `us_market`(미국 정규장 중 30분마다, 끝난 뒤 한 번): 다우·S&P 500·나스닥·원/달러 환율의 5분봉과 시가·고가·저가·종가·전일 종가(yfinance). 홈의 미국 시장 칸과 미국시장 지표 상세용이다.

In [ ]:
US_SYMBOLS = {'^DJI': '다우존스', '^GSPC': 'S&P 500', '^IXIC': '나스닥', 'KRW=X': '원/달러 환율'}


def step_night_live(T=None):
    now = now_kst()
    if not (now.hour >= 18 or now.hour < 6 or (now.hour == 6 and now.minute <= 10)):
        return '야간장 시간이 아님'
    D = (now if now.hour >= 18 else now - pd.Timedelta(days=1)).strftime('%Y%m%d')
    if pd.Timestamp(D).weekday() >= 5:
        return '주말 밤(야간장 없음)'
    code = lr.front(D)
    kis = KIS(str(ENV_FILE))
    day = [r for r in kis.daily('F', code, D, D) if r.get('stck_bsop_date') == D]
    if not day:
        return f'{D} 주간장 없음(휴장)'
    r = day[0]
    ensure_tables()
    upsert('model.futures_daily', pd.DataFrame([dict(date=D, session='day', code=code, open=float(r['futs_oprc']), high=float(r['futs_hgpr']),
                                                     low=float(r['futs_lwpr']), close=float(r['futs_prpr']), volume=float(r.get('acml_vol') or 0))]), ['date', 'session'])
    nm = lr.night_minutes(kis, D, code)
    n = upsert('model.night_minutes', nm, ['date', 'code', 'hhmmss'])
    return f'{D} 야간 1분봉 {n}개(마지막 {nm.hhmmss.max() if len(nm) else "-"})'


def step_us_market(T=None):
    import yfinance as yf
    ensure_tables()
    out = []
    for sym, name in US_SYMBOLS.items():
        m = yf.download(sym, period='5d', interval='5m', progress=False, auto_adjust=False)
        d = yf.download(sym, period='15d', interval='1d', progress=False, auto_adjust=False)
        for f in (m, d):
            f.columns = [c[0] if isinstance(c, tuple) else c for c in f.columns]
        if m.empty or d.empty:
            out.append(f'{name} 없음')
            continue
        idx = pd.to_datetime(m.index)
        m.index = (idx.tz_localize('UTC') if idx.tz is None else idx).tz_convert('America/New_York')
        last = m.index[-1].date()
        today = m[m.index.date == last]
        upsert('model.us_intraday', pd.DataFrame(dict(symbol=sym, us_date=last.strftime('%Y%m%d'), ts=[t.tz_convert('UTC') for t in today.index],
                                                     price=today.Close.astype(float).to_numpy())), ['symbol', 'ts'])
        d.index = pd.to_datetime(d.index).strftime('%Y%m%d')
        dd = d[['Open', 'High', 'Low', 'Close']].astype(float).assign(prev=lambda x: x.Close.shift(1)).dropna(subset=['prev'])
        upsert('model.us_daily', pd.DataFrame(dict(symbol=sym, us_date=dd.index, name=name, open=dd.Open, high=dd.High, low=dd.Low, close=dd.Close,
                                                  prev_close=dd.prev)).reset_index(drop=True), ['symbol', 'us_date'])
        out.append(f'{name} {last:%m-%d} {len(today)}봉')
    return ', '.join(out)

## 5-2. 장중 1분봉 (종목 상세 '1일' 그래프)
- `intraday`(거래일 장 시간에 1분마다): 고객 화면 사용자들의 보유·관심 종목(app.holdings · app.watch)의 오늘 1분봉을 KIS에서 받아 `model.intraday`에 넣는다.
- 아침 채점(`open`)과 겹치지 않게 그날 채점이 끝난 뒤부터 돈다(채점이 없는 날·채점이 실패한 날은 개장 `OPEN_WAIT_MIN`분 뒤부터). 장 마감 5분 뒤까지.
- 종목마다 처음이면(또는 중간이 비었으면) 개장부터 지금까지 거꾸로 120개씩 받고, 그 뒤로는 지금 시각까지 120개 한 번만 받는다(겹치는 줄은 덮어씀).
- 개장·마감 시각은 달력(special_days.csv)의 그날 값. 휴장일은 하지 않는다.
- `intraday_backfill`(손으로 한 번): 지난 거래일(카드가 있는 날, 최근 `INTRADAY_BACKFILL_DAYS`일)의 1분봉을 채운다. 시연 시각으로 지난 날을 볼 때 쓴다.
- 1분마다 도는 단계라 성공은 `model.runs`에 남기지 않고, 실패는 남긴다. 텔레그램은 연속 실패의 첫 번만 보낸다.

In [ ]:
INTRADAY_BACKFILL_DAYS = 20


def user_codes():
    with db() as con:
        return sorted(r[0] for r in con.execute('select code from app.holdings union select code from app.watch'))


def day_hours(T):
    '''그날 개장·마감(HHMM). 달력에 없으면 09:00~15:30'''
    day = market_day(T)
    op = (day.open if day is not None and day.open else '09:00').replace(':', '')
    cl = (day.close if day is not None and day.close else '15:30').replace(':', '')
    return op, cl


def minute_rows(kis, code, T, hhmmss):
    '''그날 hhmmss까지의 1분봉 최대 120개(최근 것부터). 그날 장 시간 안의 줄만'''
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-time-dailychartprice', 'FHKST03010230',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_HOUR_1': hhmmss, 'FID_INPUT_DATE_1': T,
                    'FID_PW_DATA_INCU_YN': 'Y', 'FID_FAKE_TICK_INCU_YN': ''})
    op, cl = day_hours(T)
    return [dict(day=T, code=code, hhmm=r['stck_cntg_hour'][:4], price=float(r['stck_prpr']), volume=float(r.get('cntg_vol') or 0))
            for r in res.get('output2') or [] if r.get('stck_bsop_date') == T and op <= r['stck_cntg_hour'][:4] <= cl]


def full_day(kis, code, T, until):
    '''개장부터 until(HHMMSS)까지 거꾸로 120개씩'''
    op, _ = day_hours(T)
    out, h = [], until
    for _ in range(6):
        rows = minute_rows(kis, code, T, h)
        out += rows
        if not rows or min(r['hhmm'] for r in rows) <= op:
            break
        first = pd.Timestamp(f"2000-01-01 {min(r['hhmm'] for r in rows)[:2]}:{min(r['hhmm'] for r in rows)[2:]}") - pd.Timedelta(minutes=1)
        h = f'{first:%H%M}00'
    return out


def intraday_ready(T):
    '''오늘 1분봉을 받을 때인지: (받을지, 이유)'''
    day = market_day(T)
    if pd.Timestamp(T).weekday() >= 5 or (day is not None and day.krx_open == '0'):   # 달력에는 주말 줄이 없다
        return False, '휴장일'
    op, cl = day_hours(T)
    now = now_kst()
    hm = f'{now:%H%M}'
    close_end = (pd.Timestamp(f'2000-01-01 {cl[:2]}:{cl[2:]}') + pd.Timedelta(minutes=5)).strftime('%H%M')
    if hm < op or hm > close_end:
        return False, '장 시간이 아님'
    with db() as con:
        checked = con.execute('select count(*) from model.checks where t = %s and checked_at is not null', (T,)).fetchone()[0]
        cards = con.execute("select count(*) from model.cards where t = %s and status = 'ok'", (T,)).fetchone()[0]
    wait_end = (pd.Timestamp(f'2000-01-01 {op[:2]}:{op[2:]}') + pd.Timedelta(minutes=OPEN_WAIT_MIN)).strftime('%H%M')
    if cards and not checked and hm < wait_end:
        return False, '아침 채점 기다림'
    return True, ''


def step_intraday(T):
    ok, why = intraday_ready(T)
    if not ok:
        return why
    codes = user_codes()
    if not codes:
        return '사용자 종목 없음'
    ensure_tables()
    op, cl = day_hours(T)
    with db() as con:
        have = {r[0]: (r[1], r[2]) for r in con.execute('select code, min(hhmm), count(*) from model.intraday where day = %s and code = any(%s) group by code', (T, codes))}
    now = now_kst()
    until = f'{min(f"{now:%H%M}", cl)}00'
    kis = KIS(str(ENV_FILE))
    def one(code):
        h = have.get(code)
        return minute_rows(kis, code, T, until) if h and h[0] <= op and h[1] >= 5 else full_day(kis, code, T, until)
    rows = sum(pmap(one, codes), [])
    n = upsert('model.intraday', pd.DataFrame(rows, columns=['day', 'code', 'hhmm', 'price', 'volume']).drop_duplicates(['day', 'code', 'hhmm']), ['day', 'code', 'hhmm'])
    return f'{T} {len(codes)}종목 1분봉 {n}줄(~{until[:2]}:{until[2:4]})'


def step_intraday_backfill(T=None):
    ensure_tables()
    codes = user_codes()
    with db() as con:
        days = [r[0] for r in con.execute("select distinct t from model.cards where status = 'ok' order by t desc limit %s", (INTRADAY_BACKFILL_DAYS,))]
    kis = KIS(str(ENV_FILE))
    total = 0
    for T in sorted(days):
        if T >= today():
            continue
        _, cl = day_hours(T)
        rows = sum(pmap(lambda c: full_day(kis, c, T, f'{cl}00'), codes), [])
        total += upsert('model.intraday', pd.DataFrame(rows, columns=['day', 'code', 'hhmm', 'price', 'volume']).drop_duplicates(['day', 'code', 'hhmm']), ['day', 'code', 'hhmm'])
        print(' ', T, len(rows), '줄', flush=True)
    return f'{len(days)}일 · {len(codes)}종목 1분봉 {total}줄'

## 5-3. 시세 요약 기록 (종목 상세 '시세 요약')
- 고객 화면 종목 상세의 거래량 · 시가총액 · 외국인 보유 · 52주 최고/최저에 쓴다. 대상은 사용자들의 보유·관심 종목(1분봉과 같음).
- **날짜별로 남긴다**: 시연 시각으로 지난 날을 볼 때 그 날 전 거래일 값이 나오게(웹은 일봉과 같은 기준으로 그날 전까지만 읽음).
- `model.stock_quote_daily`(종목·날짜): KIS 일봉(`FHKST03010100`, 시장 J)의 장중 고가·저가·종가·거래량 + 일자별 시세(`FHKST01010400`)의 외국인 소진율.
  - 웹은 52주 최고/최저를 이 표의 1년치 고가·저가에서 그 날 기준으로 계산한다(장중 기준).
  - 외국인 소진율은 KIS가 최근 30거래일만 주므로 그보다 앞 날짜는 비어 있다.
- `model.stock_info`(종목): 현재가 조회(`FHKST01010100`)의 상장 주식 수와 외국인 한도 비율(= 보유 수량 ÷ 상장 주식 수 ÷ 소진율).
  - 시가총액 = 그날 종가 × 상장 주식 수(오늘 주식 수라 그사이 증자 등이 있었으면 근삿값). 외국인 보유 비율 = 소진율 × 한도 비율.
- `quote`: 최근 `QUOTE_RECENT_DAYS`일을 다시 받아 덮어쓴다. 장 마감 종가 단계(`close_price`, 15:45)가 끝에 부른다(실패해도 종가 저장은 그대로).
  20:30 `close`에서는 부르지 않는다: 16:00 뒤에는 KIS 일봉의 그날 종가·거래량 칸에 시간외 거래 값이 들어가 맞는 줄을 덮어쓰게 된다(2026-11-25).
- `quote_backfill`(손으로 한 번): 1년치(`QUOTE_BACKFILL_DAYS`일)를 채운다. 새 종목이 생기면 `quote`가 그 종목만 1년치를 받는다.
- `fill_new`(10분마다, 2026-11-25): 사용자가 새로 넣은 종목만 시세 요약 1년치와 최근 `INTRADAY_BACKFILL_DAYS`거래일 1분봉('1일'·'1주'·'1달' 그래프)을 채운다.
  종목을 추가하면 늦어도 10분 안에 종목 상세가 채워진다(웹은 KIS 열쇠가 없어 서버가 알아채서 받는다). 새 종목이 없으면 DB 조회 한 번으로 끝난다.
- 오늘 줄은 그날 정규장 마감 10분 뒤부터 넣는다(그 전엔 장이 끝나지 않음).


In [ ]:
QUOTE_RECENT_DAYS = 10       # quote가 다시 받는 최근 기간(일)
QUOTE_BACKFILL_DAYS = 370    # quote_backfill · 새 종목이 받는 기간(일, 52주 + 여유)

QUOTE_DDL = """
create table if not exists model.stock_quote_daily (code text, date text, high float8, low float8, close float8, volume float8, frgn_ehrt float8, primary key (code, date));
create table if not exists model.stock_info (code text primary key, shares float8, frgn_limit float8, updated_at timestamptz);
"""


def quote_daily(kis, code, start, end):
    """start~end(YYYYMMDD) 일봉. KIS가 한 번에 100일까지라 끝에서부터 거꾸로 나눠 받는다"""
    out, hi = {}, end
    for _ in range(6):
        res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                       {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': start, 'FID_INPUT_DATE_2': hi,
                        'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
        rows = [r for r in res.get('output2') or [] if r.get('stck_bsop_date')]
        for r in rows:
            out[r['stck_bsop_date']] = dict(code=code, date=r['stck_bsop_date'], high=float(r['stck_hgpr']), low=float(r['stck_lwpr']),
                                            close=float(r['stck_clpr']), volume=float(r.get('acml_vol') or 0))
        first = min((r['stck_bsop_date'] for r in rows), default=None)
        if not first or first <= start or len(rows) < 90:
            break
        hi = (pd.Timestamp(first) - pd.Timedelta(days=1)).strftime('%Y%m%d')
    return out


def quote_one(kis, code, start, end):
    """한 종목: 날짜별 줄(외국인 소진율 붙임)과 종목 정보"""
    days = quote_daily(kis, code, start, end)
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-price', 'FHKST01010400',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
    for r in res.get('output') or []:
        d = r.get('stck_bsop_date')
        if d in days and r.get('hts_frgn_ehrt') not in (None, ''):
            days[d]['frgn_ehrt'] = float(r['hts_frgn_ehrt'])
    o = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-price', 'FHKST01010100',
                 {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code}).get('output') or {}
    shares = float(o.get('lstn_stcn') or 0) or None
    held, ehrt = float(o.get('frgn_hldn_qty') or 0), float(o.get('hts_frgn_ehrt') or 0)
    limit = (held / shares * 100) / ehrt if shares and ehrt > 0 else None   # 한도 비율(보통 1.0, 통신주 등은 0.49 근처)
    info = dict(code=code, shares=shares, frgn_limit=round(limit, 4) if limit else None, updated_at=now_kst().to_pydatetime())
    return list(days.values()), info


def save_quotes(T, codes, days_back):
    _, cl = day_hours(T)
    closed = now_kst() >= pd.Timestamp(f'{T[:4]}-{T[4:6]}-{T[6:]} {cl[:2]}:{cl[2:]}', tz=KST) + pd.Timedelta(minutes=10)
    cut = (pd.Timestamp(T) + pd.Timedelta(days=1)).strftime('%Y%m%d') if T == today() and closed else T   # 오늘 줄은 정규장 마감 10분 뒤부터
    end = (pd.Timestamp(cut) - pd.Timedelta(days=1)).strftime('%Y%m%d')
    with db() as con:
        con.execute(QUOTE_DDL)
        have = {r[0] for r in con.execute('select distinct code from model.stock_quote_daily where code = any(%s)', (codes,))}
    kis = KIS(str(ENV_FILE))
    def one(code):
        back = days_back if code in have else QUOTE_BACKFILL_DAYS   # 처음 보는 종목은 1년치
        return quote_one(kis, code, (pd.Timestamp(end) - pd.Timedelta(days=back)).strftime('%Y%m%d'), end)
    got = pmap(one, codes, label='시세 요약')
    rows = [r for d, _ in got for r in d]
    cols = ['code', 'date', 'high', 'low', 'close', 'volume', 'frgn_ehrt']
    n = upsert('model.stock_quote_daily', pd.DataFrame(rows, columns=cols).drop_duplicates(['code', 'date']), ['code', 'date'])
    upsert('model.stock_info', pd.DataFrame([i for _, i in got if i['shares']]), ['code'])
    return f'{len(codes)}종목 {n}줄(~{end})'


def step_quote(T):
    codes = user_codes()
    return save_quotes(T, codes, QUOTE_RECENT_DAYS) if codes else '사용자 종목 없음'


def step_quote_backfill(T=None):
    return save_quotes(T or today(), user_codes(), QUOTE_BACKFILL_DAYS)


def step_fill_new(T=None):
    """새로 추가된 사용자 종목만: 시세 요약 1년치(model.stock_quote_daily가 없는 종목)와 최근 거래일 1분봉(지난 거래일 1분봉이 없는 종목)"""
    codes = user_codes()
    if not codes:
        return '사용자 종목 없음'
    with db() as con:
        con.execute(QUOTE_DDL)
        days = [r[0] for r in con.execute("select distinct t from model.cards where status = 'ok' and t < %s order by t desc limit %s", (today(), INTRADAY_BACKFILL_DAYS))]
        have_q = {r[0] for r in con.execute('select distinct code from model.stock_quote_daily where code = any(%s)', (codes,))}
        # 오늘 장중 1분봉은 intraday 단계가 새 종목도 받으므로, '지난 거래일' 1분봉이 있는지로 본다
        have_i = {r[0] for r in con.execute('select distinct code from model.intraday where code = any(%s) and day = %s', (codes, days[0] if days else ''))}
    new_q = [c for c in codes if c not in have_q]
    new_i = [c for c in codes if c not in have_i] if days else []
    if not new_q and not new_i:
        return '새 종목 없음'
    out = []
    if new_q:
        out.append('시세 요약 ' + save_quotes(today(), new_q, QUOTE_BACKFILL_DAYS))
    if new_i:
        kis = KIS(str(ENV_FILE))
        total = 0
        for D in sorted(days):
            _, cl = day_hours(D)
            rows = sum(pmap(lambda c: full_day(kis, c, D, f'{cl}00'), new_i), [])
            total += upsert('model.intraday', pd.DataFrame(rows, columns=['day', 'code', 'hhmm', 'price', 'volume']).drop_duplicates(['day', 'code', 'hhmm']), ['day', 'code', 'hhmm'])
        out.append(f'1분봉 {len(new_i)}종목 · {len(days)}거래일 {total}줄')
    return f'새 종목 {sorted(set(new_q) | set(new_i))}: ' + ' · '.join(out)


## 5-4. 장 마감 종가 (2026-11-25)
종가는 **정규장 공식 종가**(마감 동시호가 체결가, 다음 날 기준가)여야 한다. 카드의 '어제 종가'와 채점(시가 ÷ 어제 종가)이 이 값을 쓴다.
- 문제: KIS 일봉의 '그날 종가' 칸은 16:00 뒤 시간외 거래 동안 그 체결가로 바뀌었다가 밤사이에 공식 종가로 돌아온다. 20:10(팀원 `run_day.sh`)·20:30(`close`)에 받은 값도 시간외 가격이었다(11/20·23·24, 955종목 중 86~90%가 틀림).
  자료 이어 받기는 이미 있는 날짜·종목 줄을 덮어쓰지 않으므로 처음 들어간 값이 그대로 남는다.
- `close_price`(평일 15:45, 수능일처럼 늦게 닫는 날은 달력의 마감 + `CLOSE_AFTER_MIN`분까지 기다림): 그날 종가를 한 번 받아 아침 자료 폴더(`live/<T>`)와 `model.stock_daily`에 넣는다.
  - 마감 봉(마감 시각 1분봉 = 마감 동시호가 체결가)이 있으면 그 가격, 없으면 그 시각 일봉 종가 칸. 둘 다 있는데 다르면 마감 봉 값을 쓰고 텔레그램으로 알린다.
  - 실제로는 15:45에 마감 봉이 없다: KIS는 마감 동시호가 결과를 16시 무렵에야 1분봉으로 내준다(2026-11-25: 15:42·15:57엔 955종목 모두 15:19 봉이 마지막, 16:12엔 618종목에 생겼고 모두 공식 종가와 같음). 그래서 15:45엔 일봉 종가 칸을 쓰고, 이 시각 그 칸은 공식 종가다(11-25 한국거래소 목록과 955종목 모두 같음, 16:12엔 534종목이 시간외 가격으로 바뀜).
  - 저장한 종가를 한국거래소 종목 목록(FinanceDataReader)의 종가와 한 번 더 대조한다. 다르면 알림만 보내고 저장 값은 바꾸지 않는다(그 시각 목록이 확정 값인지는 아직 확인 중).
  - 그날 20:30 `close`는 이 폴더를 이어받고 이미 있는 줄을 덮어쓰지 않으므로 맞는 종가가 지켜진다(팀원 고정 코드는 그대로).
  - 같은 단계 끝에서: 사용자 종목의 마감 무렵 1분봉을 `model.intraday`에 채우고, 시세 요약(`quote`, 5-3)을 받는다. 이때는 마감 봉이 없어 '1일' 그래프가 15:19에서 끝나므로, 20:30 `close`에서 마감 봉을 한 번 더 받아 채운다('1일' 그래프 끝 = 공식 종가).
- 아침 검사(`morning0730` 안, 카드 만들기 전): 폴더의 어제 종가를 그 시각 일봉 종가 칸과 먼저 비교하고(종목마다 한 번), 다른 종목만 어제 마감 봉으로 확인한다. **마감 봉으로 확인된 종목만 바로잡고**, 마감 봉이 없는 종목은 바꾸지 않고 알림만 보낸다(아침에도 일봉 칸이 시간외 가격일 수 있어 그 값으로 덮어쓰면 맞는 종가를 틀리게 만든다, 2026-11-25). 그 뒤 카드를 만든다.
- `close_fix`(손으로): 지난 날짜 종가를 바로잡고, 그 종가를 '어제 종가'로 쓴 다음 거래일 카드(어제 종가 · 08:10 장전 변화)와 채점을 다시 계산한다. 공개한 예측(H · 범위 · 믿을 정도)은 바꾸지 않는다. 바꾸기 전 값은 `~/overnight/backup/close_fix_<날짜>/`.


In [ ]:
CLOSE_AFTER_MIN = 15   # 종가 받는 시각 = 그날 정규장 마감 + 15분(16:00 시간외 단일가 전)


def day_daily(kis, code, T):
    """그날 일봉 줄(시가 > 0)만. 없으면 None"""
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': T, 'FID_INPUT_DATE_2': T,
                    'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
    return next((x for x in res.get('output2') or [] if x.get('stck_bsop_date') == T and float(x.get('stck_oprc') or 0) > 0), None)


def day_close(kis, code, T, bars_out=None):
    """그날 정규장 종가. (시가, 종가, 일봉 칸과 마감 봉이 달랐는지, 마감 봉이 있었는지). 그날 거래가 없으면(시가 0) None.
    마감 봉 = 마감 시각(보통 15:30) 1분봉 = 마감 동시호가 체결가 = 공식 종가. KIS는 이 봉을 16시 무렵에야 내준다
    (2026-11-25 확인: 15:42·15:57엔 955종목 모두 15:19 봉이 마지막, 16:12엔 618종목에 생겼고 모두 공식 종가와 같음).
    마감 봉이 없으면 일봉 종가 칸을 쓴다(15:45엔 이 칸이 공식 종가, 16:00 시간외 거래가 시작되면 바뀐다).
    bars_out(목록)을 주면 받은 마감 무렵 1분봉 줄을 model.intraday 모양으로 덧붙인다"""
    res = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-daily-itemchartprice', 'FHKST03010100',
                   {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_DATE_1': T, 'FID_INPUT_DATE_2': T,
                    'FID_PERIOD_DIV_CODE': 'D', 'FID_ORG_ADJ_PRC': '0'})
    r = next((x for x in res.get('output2') or [] if x.get('stck_bsop_date') == T and float(x.get('stck_oprc') or 0) > 0), None)
    if r is None:
        return None
    op, dc = float(r['stck_oprc']), float(r['stck_clpr'])
    _, cl = day_hours(T)
    m = kis_call(kis, '/uapi/domestic-stock/v1/quotations/inquire-time-dailychartprice', 'FHKST03010230',
                 {'FID_COND_MRKT_DIV_CODE': 'J', 'FID_INPUT_ISCD': code, 'FID_INPUT_HOUR_1': cl + '00', 'FID_INPUT_DATE_1': T,
                  'FID_PW_DATA_INCU_YN': 'N', 'FID_FAKE_TICK_INCU_YN': ''})
    bars = [x for x in m.get('output2') or [] if x.get('stck_bsop_date') == T and x['stck_cntg_hour'][:4] <= cl]
    if bars_out is not None:
        op_hm, _ = day_hours(T)
        bars_out += [dict(day=T, code=code, hhmm=x['stck_cntg_hour'][:4], price=float(x['stck_prpr']), volume=float(x.get('cntg_vol') or 0))
                     for x in bars if x['stck_cntg_hour'][:4] >= op_hm]
    last = max(bars, key=lambda x: x['stck_cntg_hour']) if bars else None
    if last is not None and last['stck_cntg_hour'][:4] == cl:   # 마감 동시호가 봉 = 공식 종가
        mc = float(last['stck_prpr'])
        return op, mc, mc != dc, True
    return op, dc, False, False


def put_closes(folder, rows, overwrite):
    """자료 폴더의 universe_daily.csv와 model.stock_daily에 종가 줄을 넣는다. overwrite=False면 없는 줄만(자료 이어 받기와 같은 규칙)"""
    f = Path(folder) / 'data/universe_daily.csv'
    ud = pd.read_csv(f, dtype=lp.TYPES)
    new = pd.DataFrame(rows, columns=['date', 'Open', 'Close', 'code'])
    key = ['date', 'code']
    if overwrite:
        ud = ud.set_index(key)
        hit = new.set_index(key)
        hit = hit[hit.index.isin(ud.index)]
        ud.loc[hit.index, 'Close'] = hit.Close
        ud = ud.reset_index()[['date', 'Open', 'Close', 'code']]
        put = new[new.set_index(key).index.isin(hit.index)]
    else:
        put = new[~new.set_index(key).index.isin(ud.set_index(key).index)]
        ud = pd.concat([ud, put], ignore_index=True).sort_values(['code', 'date'])
    ud.to_csv(f, index=False)
    upsert('model.stock_daily', put.rename(columns={'Open': 'open', 'Close': 'close'})[['code', 'date', 'open', 'close']], ['code', 'date'])
    return len(put)


def step_close_price(T):
    day = market_day(T)
    if pd.Timestamp(T).weekday() >= 5 or (day is not None and day.krx_open == '0'):
        return '휴장일'
    _, cl = day_hours(T)
    at = pd.Timestamp(f'{T[:4]}-{T[4:6]}-{T[6:]} {cl[:2]}:{cl[2:]}', tz=KST) + pd.Timedelta(minutes=CLOSE_AFTER_MIN)
    if T == today() and now_kst() < at:
        print('종가 받는 시각', f'{at:%H:%M}', '까지 기다림', flush=True)
        time.sleep((at - now_kst()).total_seconds())
    folder = LIVE / T
    if not (folder / 'data/universe_daily.csv').exists():
        raise RuntimeError(f'{folder.name} 자료 폴더가 없어 종가를 넣을 곳이 없습니다(아침 단계 확인)')
    codes = sorted(pd.read_csv(folder / 'data/universe_daily.csv', dtype=lp.TYPES).code.unique())
    kis = KIS(str(ENV_FILE))
    users = set(user_codes())
    bars = []   # 사용자 종목의 마감 무렵 1분봉('1일' 그래프 끝을 공식 종가로)
    got = dict(zip(codes, pmap(lambda c: day_close(kis, c, T, bars if c in users else None), codes, label='장 마감 종가')))
    rows = [dict(date=T, Open=v[0], Close=v[1], code=c) for c, v in got.items() if v]
    diff = [c for c, v in got.items() if v and v[2]]
    nbar = sum(1 for v in got.values() if v and v[3])
    ensure_tables()
    n = put_closes(folder, rows, overwrite=False)
    if diff:
        telegram(f'[아침 모델] {T} 장 마감 종가: 일봉 칸과 마감 봉이 다른 종목 {len(diff)}개(마감 봉 값으로 저장) {", ".join(diff[:10])}')
    try:   # 한국거래소 종목 목록 종가와 대조(알림만, 값은 바꾸지 않는다). 실패해도 종가 저장은 그대로
        nk, kdiff = krx_check(T, {r['code']: r['Close'] for r in rows})
        krx = f'한국거래소 목록과 같음 {nk - len(kdiff)}/{nk}'
        if kdiff:
            telegram(f'[아침 모델] {T} 장 마감 종가: 한국거래소 목록과 다른 종목 {len(kdiff)}개(저장 값은 그대로, 확인 필요) ' + ', '.join(f'{c} {a:,.0f}≠{b:,.0f}' for c, a, b in kdiff[:5]))
    except Exception as err:
        krx = f'한국거래소 목록 대조 실패 {type(err).__name__}'

    nb_ = upsert('model.intraday', pd.DataFrame(bars, columns=['day', 'code', 'hhmm', 'price', 'volume']).drop_duplicates(['day', 'code', 'hhmm']), ['day', 'code', 'hhmm'])
    try:   # 시세 요약 기록(5-3): 정규장 값이 확정된 이때 받는다(20:30엔 시간외 값). 실패해도 종가 저장은 그대로
        quote = step_quote(T)
    except Exception as err:
        quote = f'시세 요약 실패 {type(err).__name__}: {err}'
    return (f'{T} 종가 {n}종목 저장(거래 없음 {len(codes) - len(rows)} · 마감 봉 있음 {nbar}, 없어서 일봉 칸 {len(rows) - nbar} · 둘이 다름 {len(diff)})'
            f' · {krx} · 마감 무렵 1분봉 {nb_}줄 · {quote}')


def verify_prev_close(T, dry=False):
    """아침 카드 전 검사: 폴더의 어제 종가가 지금 일봉 종가 칸과 같은지 보고, 다른 종목만 어제 마감 봉(공식 종가)으로 확인한다.
    마감 봉으로 확인된 종목만 바로잡는다. 마감 봉이 없으면 바꾸지 않고 알림만 보낸다(2026-11-25: 아침에도 일봉 칸이 시간외 가격일 수 있어,
    그 값으로 덮어쓰면 맞는 종가를 틀린 값으로 바꾸게 된다). dry=True면 저장·알림 없이 결과 문장만 돌려준다(시험용)"""
    folder = LIVE / T
    ud = pd.read_csv(folder / 'data/universe_daily.csv', dtype=lp.TYPES)
    D = ud.date[ud.date < T].max()
    have = ud[ud.date == D].set_index('code').Close
    kis = KIS(str(ENV_FILE))
    daily = dict(zip(have.index, pmap(lambda c: day_daily(kis, c, D), list(have.index), label='어제 종가 검사')))
    odd = [c for c, r in daily.items() if r is not None and abs(float(r['stck_clpr']) - have[c]) > 0.5]   # 일봉 칸과 다른 종목만 마감 봉으로 다시 확인
    got = dict(zip(odd, pmap(lambda c: day_close(kis, c, D), odd))) if odd else {}
    nobar = [c for c in odd if not (got.get(c) and got[c][3])]                                  # 마감 봉이 없어 확인 못 함
    rows = [dict(date=D, Open=v[0], Close=v[1], code=c) for c, v in got.items() if v and v[3] and abs(v[1] - have[c]) > 0.5]
    ok = len(odd) - len(nobar) - len(rows)                                                      # 일봉 칸만 다르고 마감 봉은 저장 값과 같음
    msg = f'어제({D}) 종가 {len(have)}종목 확인: 일봉 칸과 다름 {len(odd)}(마감 봉으로 맞음 {ok} · 바로잡을 것 {len(rows)} · 마감 봉 없어 그대로 둠 {len(nobar)})'
    if dry:
        return msg + (' · 바로잡을 것 ' + ', '.join(f"{r['code']} {have[r['code']]:,.0f}→{r['Close']:,.0f}" for r in rows[:5]) if rows else '')
    if rows:
        ensure_tables()
        put_closes(folder, rows, overwrite=True)
        msg += ' · 바로잡음 ' + ', '.join(f"{r['code']} {have[r['code']]:,.0f}→{r['Close']:,.0f}" for r in rows[:5])
    if rows or nobar:
        telegram('[아침 모델] ' + msg + (f" · 마감 봉 없음 {', '.join(nobar[:10])}(일봉 칸 값으로 바꾸지 않음, 확인 필요)" if nobar else ''))
    return msg


def krx_check(T, closes):
    """한국거래소 종목 목록(FinanceDataReader StockListing('KRX'))의 종가와 대조한다. (대조한 종목 수, [(종목, 저장 값, 목록 값)]).
    목록은 '오늘' 값만 주므로 T가 오늘일 때만 대조한다(2026-11-25 16:09 확인: 15:45에 저장한 955종목과 모두 같음)"""
    if T != today():
        return 0, []
    import FinanceDataReader as fdr
    k = fdr.StockListing('KRX').set_index('Code')
    both = [c for c in closes if c in k.index]
    return len(both), [(c, closes[c], float(k.loc[c, 'Close'])) for c in both if abs(float(k.loc[c, 'Close']) - closes[c]) >= 0.5]


def refill_close_bars(T, dry=False):
    """사용자 종목의 마감 봉(보통 15:30)을 다시 받아 model.intraday에 채운다. 15:45엔 KIS가 이 봉을 아직 주지 않아
    '1일' 그래프가 15:19에서 끝나므로 20:30 장 마감 단계에서 한 번 더 받는다(2026-11-25). dry=True면 저장하지 않는다"""
    users = sorted(set(user_codes()))
    if not users:
        return '마감 봉: 사용자 종목 없음'
    _, cl = day_hours(T)
    kis = KIS(str(ENV_FILE))
    bars = []
    got = dict(zip(users, pmap(lambda c: day_close(kis, c, T, bars), users)))
    last = [b for b in bars if b['hhmm'] == cl]
    if not dry and bars:
        upsert('model.intraday', pd.DataFrame(bars, columns=['day', 'code', 'hhmm', 'price', 'volume']).drop_duplicates(['day', 'code', 'hhmm']), ['day', 'code', 'hhmm'])
    return f"마감 봉 {len({b['code'] for b in last})}/{sum(1 for v in got.values() if v)}종목"


def fix_records(cards, checks):
    """카드(cards 날짜)의 어제 종가·08:10 장전 변화와 채점(checks 날짜)을 model.stock_daily 종가 기준으로 다시 계산. 예측 값(h·범위·등급)은 그대로"""
    with db() as con, con.cursor() as cur:
        cur.execute("""create temp table fixpc on commit drop as
          select c.t, c.slot, c.code, c.pc as pc_old,
                 (select d.close from model.stock_daily d where d.code = c.code and d.date < c.t order by d.date desc limit 1) as pc_new
          from model.card_stocks c where c.t = any(%s)""", (cards,))
        cur.execute("""update model.card_stocks c set pc = f.pc_new,
                         nxt_g = case when c.nxt_g is null then null else (1 + c.nxt_g) * f.pc_old / f.pc_new - 1 end
                       from fixpc f where c.t = f.t and c.slot = f.slot and c.code = f.code and f.pc_new is not null and f.pc_new <> f.pc_old""")
        n_card = cur.rowcount
        cur.execute("""update model.check_stocks k set y = k.open / c.pc - 1,
                         in80 = (k.open / c.pc - 1) between c.l80 and c.u80, in90 = (k.open / c.pc - 1) between c.l90 and c.u90,
                         above = (k.open / c.pc - 1) > c.u80, below = (k.open / c.pc - 1) < c.l80
                       from model.card_stocks c where c.t = k.t and c.slot = k.slot and c.code = k.code and k.t = any(%s)""", (checks,))
        cur.execute("""with s as (
            select k.t, k.slot, round(avg(k.in80::int) * 100, 1) b80, round(avg(k.in90::int) * 100, 1) b90,
                   round((avg(abs(k.y - c.h)) * 100)::numeric, 3) mae,
                   round(avg((sign(c.nxt_g) = sign(k.y))::int) filter (where c.grade_level is not null and c.nxt_g <> 0 and k.y <> 0) * 100, 1) dir
            from model.check_stocks k join model.card_stocks c using (t, slot, code) where k.t = any(%s) group by k.t, k.slot)
          update model.checks ch set in_band80 = s.b80, in_band90 = s.b90, mae_h = s.mae, dir_nxt_all = s.dir,
                 summary = ch.summary || jsonb_build_object('in_band80', s.b80, 'in_band90', s.b90, 'mae_H', s.mae, 'dir_nxt_all', s.dir,
                                                            'close_fix', '어제 종가를 공식 종가로 바로잡아 다시 계산(예측 값은 그대로)')
          from s where ch.t = s.t and ch.slot = s.slot""", (checks,))
        n_check = cur.rowcount
    return n_card, n_check


def step_close_fix(D):
    """D(지난 거래일)의 종가를 공식 종가로 바로잡고, D를 '어제'로 쓴 다음 거래일 카드와 채점을 다시 계산한다(손으로)"""
    bk = MODEL.parent / f'backup/close_fix_{D}'
    bk.mkdir(parents=True, exist_ok=True)
    folders = [p for p in sorted(LIVE.iterdir()) if (p / 'data/universe_daily.csv').exists()]
    base = pd.read_csv(folders[-1] / 'data/universe_daily.csv', dtype=lp.TYPES)
    codes = sorted(base[base.date == D].code.unique())
    kis = KIS(str(ENV_FILE))
    got = dict(zip(codes, pmap(lambda c: day_close(kis, c, D), codes, label='종가 바로잡기')))
    rows = [dict(date=D, Open=v[0], Close=v[1], code=c) for c, v in got.items() if v]
    with db() as con:
        nxt = con.execute('select min(t) from model.cards where t > %s', (D,)).fetchone()[0]
        for tb, col, day in [('stock_daily', 'date', D), ('card_stocks', 't', nxt), ('check_stocks', 't', nxt), ('checks', 't', nxt)]:
            cur = con.execute(f'select * from model.{tb} where {col} = %s', (day,))
            pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]).to_csv(bk / f'{tb}_before.csv', index=False)
    changed = 0
    for p in folders:
        if (pd.read_csv(p / 'data/universe_daily.csv', dtype=lp.TYPES).date == D).any():
            shutil.copy2(p / 'data/universe_daily.csv', bk / f'{p.name}_universe_daily.csv')
            changed = put_closes(p, rows, overwrite=True)
    n_card, n_check = fix_records([nxt], [nxt]) if nxt else (0, 0)
    return f'{D} 종가 {changed}종목 확인·저장, 다음 카드 {nxt} 종목 줄 {n_card}개 · 채점 요약 {n_check}개 다시 계산(백업 {bk})'


## 6. 실행 묶음 (서버 일정이 부르는 곳)
`run_step(이름, 날짜)`가 한 단계를 돌리고 `model.runs`에 기록한다. 같은 단계가 겹쳐 돌지 않게 잠그고, 실패하면 텔레그램으로 알린다(`ALERT_TELEGRAM_TOKEN`, `ALERT_TELEGRAM_CHAT_ID`가 있을 때).
`rebuild_from_db(이름)`: 서버 자료 폴더가 없어졌을 때 DB의 원자료로 다시 만든다(미국 시간봉·VIX·코스피200 긴 표는 다시 내려받는다).

In [ ]:
def telegram(text):
    tok, chat = ENV.get('ALERT_TELEGRAM_TOKEN'), ENV.get('ALERT_TELEGRAM_CHAT_ID')
    if not tok or not chat:
        return
    try:
        data = urllib.parse.urlencode(dict(chat_id=chat, text=text[:3500])).encode()
        urllib.request.urlopen(f'https://api.telegram.org/bot{tok}/sendMessage', data=data, timeout=20)
    except Exception:
        pass


def log_run(step, T, started, ok, message):
    try:
        ensure_tables()
        with db() as con:
            con.execute('insert into model.runs (step, t, started_at, finished_at, ok, message) values (%s, %s, %s, now(), %s, %s)',
                        (step, T, started.to_pydatetime(), ok, message[:4000]))
    except Exception as err:
        print('실행 기록 실패', type(err).__name__, flush=True)


STEPS = {'morning0730': lambda T: step_morning(T, '0730'), 'morning0811': lambda T: step_morning(T, '0811'),
         'open': step_open, 'close': step_close,
         'bootstrap': lambda T: str(bootstrap(T)), 'save_boot': lambda T: str(save_folder_db(LIVE / BOOT_NAME)),
         'bootstrap_rest': lambda T: str(bootstrap(T, steps={'premarket', 'market', 'dart'})),
         'night_live': step_night_live, 'us_market': step_us_market,   # 앞 단계가 끝난 뒤 나머지만
         'backfill': step_backfill,   # 지난 거래일 카드·채점 채우기(손으로 한 번)
         'intraday': step_intraday, 'intraday_backfill': step_intraday_backfill,
         'quote': step_quote, 'quote_backfill': step_quote_backfill,   # 시세 요약 기록(5-3)
         'close_price': step_close_price, 'close_fix': step_close_fix,   # 장 마감 종가(5-4)
         'fill_new': step_fill_new}   # 새로 넣은 종목 자료 채우기(5-3)
QUIET = {'intraday', 'fill_new'}   # 1분·10분마다 도는 단계: 성공은 기록하지 않고, 텔레그램은 연속 실패의 첫 번만


def run_step(step, T=None):
    T = T or (last_finished_day() if step.startswith('bootstrap') else today())   # 처음 자료는 끝난 장까지만
    LIVE.mkdir(parents=True, exist_ok=True)
    lock = open(LIVE / f'.lock_{step}', 'w')
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        print(step, '이미 도는 중이라 건너뜀', flush=True)
        return
    started = now_kst()
    print(f'=== {step} {T} 시작 {started:%H:%M:%S}', flush=True)
    try:
        msg = STEPS[step](T)
        print(f'=== {step} {T} 끝 {now_kst():%H:%M:%S} {msg}', flush=True)
        if step in QUIET:
            (LIVE / f'.fail_{step}').unlink(missing_ok=True)
        else:
            log_run(step, T, started, True, str(msg))
    except BaseException as err:
        text = f'{type(err).__name__}: {err}\n' + traceback.format_exc()[-1500:]
        print(text, flush=True)
        log_run(step, T, started, False, text)
        mark = LIVE / f'.fail_{step}'
        if step not in QUIET or not mark.exists():
            telegram(f'[아침 모델] {step} {T} 실패: {type(err).__name__}: {str(err)[:300]}')
        if step in QUIET:
            mark.touch()
        raise SystemExit(1)


def rebuild_from_db(name=BOOT_NAME, until=None):
    """DB 원자료로 자료 폴더를 다시 만든다."""
    out, data = LIVE / name, LIVE / name / 'data'
    data.mkdir(parents=True, exist_ok=True)
    q = lambda sql: pd.read_sql(sql, psycopg.connect(db_url()))
    save_csv(model_universe(), data / 'universe.csv')
    for n in ('fred_release_dates.csv', 'fomc_dates.csv'):
        shutil.copy2(MODEL / 'data' / n, data / n)
    fut = q('select date, session, code, open, high, low, close, volume from model.futures_daily order by date, session')
    save_csv(fut, data / 'futures_daily.csv')
    save_csv(q('select date, code, hhmmss, price, volume from model.night_minutes order by date, hhmmss'), data / 'night_minutes.csv')
    save_csv(q('select date, open as "Open", close as "Close", code from model.stock_daily order by code, date'), data / 'universe_daily.csv')
    save_csv(q('select day, code, hhmmss, price, volume from model.premarket order by day, code, hhmmss'), data / 'universe_premarket.csv')
    save_csv(pd.DataFrame(columns=['date', 'day', 'code', 'hhmmss', 'price', 'volume']), data / 'nxt_premarket.csv')
    k = q('select date, open as "Open", high as "High", low as "Low", close as "Close", volume as "Volume" from model.ks200 order by date')
    save_csv(k.assign(code='KS200', name='코스피200 지수'), data / 'kr_daily.csv')
    _, long = boot_ks200()
    save_csv(long, data / 'ks200_long.csv')
    import collect
    save_csv(collect.collect_us_hourly(), data / 'us_hourly.csv')
    save_csv(boot_vix(), data / 'vix_daily.csv')
    save_csv(investor_placeholder(), data / 'investor_daily.csv')
    save_csv(q('select code, corp_code, rcept_no, rcept_dt, report_nm, corp_cls, flr_nm, rm from model.filings order by code, rcept_dt, rcept_no'), out / 'dart_filings.csv')
    until = until or fut[fut.session == 'day'].date.max()
    log = dict(started_at=now_kst().isoformat(timespec='seconds'), kind='rebuild_from_db', base='DB',
               trading_day=dict(date=until, open=True, checked_by='rebuild_from_db'),
               inputs_sha256={n: lp.sha(data / n) for n in lp.INPUTS}, finished_at=now_kst().isoformat(timespec='seconds'))
    (out / 'refresh_manifest.json').write_text(json.dumps(log, ensure_ascii=False, indent=1))
    return out

## 7. 손으로 돌릴 때 (서버 일정은 이 칸을 건너뛴다)

In [ ]:
# 처음 한 번: 자료 받기 → DB에 넣기
# run_step('bootstrap'); run_step('save_boot')
# 하루 단계를 손으로: run_step('morning0730', '20261121')